In [ ]:
# !hf download google/byt5-base --local-dir /root/byt5/byt5-base

In [ ]:
# !pip install sacrebleu
# !pip install transformers==4.57.1  # new version does not fit the previously released model

# Fine-tune the ByT5-Base model

import os
import gc
import re
import time
import math
import random
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

import sacrebleu


# =====================================================
# Config
# =====================================================
class Config:
    MODEL_NAME = "/root/byt5/byt5-base"
    MAX_LENGTH = 512

    EPOCHS = 10
    LEARNING_RATE = 1e-4

    OUTPUT_DIR = "/root/autodl-tmp/byt5-base-akkadian/full_aug_baseline"
    
    INPUT_DIR = "/root"
    TRAIN_FILE = "train.csv"

    PREFIX = "translate Akkadian to English: "

    # -------------------------
    # Data augmentation control
    # -------------------------
    KEEP_ORIGINAL = True
    USE_SENTENCE_ALIGNER = True
    AUGMENT_RATIO = 0.30   # Add one augmented variant for about 30% of examples

    # train setting
    PER_DEVICE_TRAIN_BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 16

    SEED = 42


# =====================================================
# Utilities
# =====================================================
def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# =====================================================
# Sentence reconstruction
# =====================================================
def reconstruct_sentence_pairs(dataframe):
    reconstructed_rows = []

    for _, row in dataframe.iterrows():
        source_text = str(row["transliteration"])
        target_text = str(row["translation"])

        target_sentences = [sentence.strip() for sentence in re.split(r'(?<=[.!?])\s+', target_text) if sentence.strip()]
        source_lines = [line.strip() for line in source_text.split('\n') if line.strip()]

        if len(target_sentences) > 1 and len(target_sentences) == len(source_lines):
            for source_segment, target_segment in zip(source_lines, target_sentences):
                if len(source_segment) > 3 and len(target_segment) > 3:
                    reconstructed_rows.append({
                        "transliteration": source_segment,
                        "translation": target_segment
                    })
        else:
            reconstructed_rows.append({
                "transliteration": source_text,
                "translation": target_text
            })

    return pd.DataFrame(reconstructed_rows)


# =====================================================
# Weak augmentation for source text only
# =====================================================
def augment_transliteration_noise(text,
                                  space_prob=0.30,
                                  hyphen_prob=0.20,
                                  case_flip_prob=0.10,
                                  short_token_drop_prob=0.08):
    """
    Apply weak source-side noise without changing the translation target.
    """
    text = str(text)

    # Normalize whitespace.
    if random.random() < space_prob:
        text = re.sub(r"\s+", " ", text).strip()

    # Perturb hyphen spacing.
    if random.random() < hyphen_prob:
        text = re.sub(r"\s*-\s*", "-", text)
        if random.random() < 0.30:
            text = text.replace("-", " - ")

    # Rarely flip character case.
    if random.random() < case_flip_prob:
        chars = list(text)
        new_chars = []
        for ch in chars:
            if ch.isalpha() and random.random() < 0.03:
                ch = ch.upper() if ch.islower() else ch.lower()
            new_chars.append(ch)
        text = "".join(new_chars)

    # Rarely drop one short token.
    if random.random() < short_token_drop_prob:
        tokens = text.split()
        if len(tokens) >= 4:
            candidate_indices = [i for i, token in enumerate(tokens) if 1 <= len(token) <= 3]
            if len(candidate_indices) > 0:
                drop_index = random.choice(candidate_indices)
                tokens.pop(drop_index)
                text = " ".join(tokens)

    text = re.sub(r"\s+", " ", text).strip()
    return text


# =====================================================
# Build training examples
# =====================================================
def build_training_examples(dataframe,
                                keep_original=True,
                                use_sentence_aligner=True,
                                augment_ratio=0.30):
    """
    Build training examples from originals, reconstructed sentence pairs,
    and weak source-side augmentation. Targets are kept unchanged.
    """
    # Collect deterministic examples before stochastic augmentation.
    parts = []

    # Keep original pairs as a stable training anchor.
    if keep_original:
        original_examples = dataframe[["transliteration", "translation"]].copy()
        parts.append(original_examples)

    # Add sentence-level pairs when the heuristic reconstruction is confident.
    if use_sentence_aligner:
        reconstructed_df = reconstruct_sentence_pairs(dataframe)
        parts.append(reconstructed_df)

    # Defensive fallback for disabled data switches.
    if len(parts) == 0:
        parts.append(dataframe[["transliteration", "translation"]].copy())

    # Deduplicate deterministic examples before stochastic augmentation.
    base_examples = pd.concat(parts, ignore_index=True)
    base_examples = base_examples.drop_duplicates().reset_index(drop=True)

    # Add one weak source-side variant to roughly augment_ratio of examples.
    augmented_rows = []
    for _, row in base_examples.iterrows():
        if random.random() < augment_ratio:
            source_text = str(row["transliteration"])
            target_text = str(row["translation"])

            # Apply lightweight source-only perturbations.
            augmented_source_text = augment_transliteration_noise(source_text)

            # Skip empty and no-op augmentations.
            if augmented_source_text and augmented_source_text != source_text:
                augmented_rows.append({
                    "transliteration": augmented_source_text,
                    "translation": target_text
                })

    # Append augmented rows and deduplicate again.
    if len(augmented_rows) > 0:
        augmented_examples = pd.DataFrame(augmented_rows)
        training_examples = pd.concat([base_examples, augmented_examples], ignore_index=True)
        training_examples = training_examples.drop_duplicates().reset_index(drop=True)
    else:
        training_examples = base_examples

    return training_examples


# =====================================================
# Tokenization
# =====================================================
def tokenize_translation_batch(batch):
    source_inputs = [Config.PREFIX + str(text) for text in batch["transliteration"]]
    target_texts = [str(text) for text in batch["translation"]]

    model_inputs = tokenizer(
        source_inputs,
        max_length=Config.MAX_LENGTH,
        truncation=True
    )
    labels = tokenizer(
        target_texts,
        max_length=Config.MAX_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# =====================================================
# Metrics
# =====================================================
def compute_translation_metrics(eval_predictions):
    predictions, labels = eval_predictions

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_predictions = [prediction.strip() for prediction in decoded_predictions]
    decoded_labels = [label.strip() for label in decoded_labels]

    bleu = sacrebleu.corpus_bleu(decoded_predictions, [decoded_labels])
    chrf = sacrebleu.corpus_chrf(decoded_predictions, [decoded_labels], word_order=2)

    score = math.sqrt(max(bleu.score, 0.0) * max(chrf.score, 0.0))

    return {
        "score": score,
        "bleu": bleu.score,
        "chrf": chrf.score,
    }


# =====================================================
# Model perturbation
# =====================================================
@torch.no_grad()
def perturb_decoder_tail_layers(model, n=2, std_ratio=0.01, include_layer_norm=False):
    target_layers = model.decoder.block[-n:]

    for layer in target_layers:
        for name, param in layer.named_parameters():
            if not param.requires_grad:
                continue

            if (not include_layer_norm) and ("layer_norm" in name.lower()):
                continue

            param_std = param.std().item()

            if param_std == 0 or np.isnan(param_std):
                continue

            noise = torch.randn_like(param) * (param_std * std_ratio)
            param.add_(noise)


def load_model_with_strategy(strategy):
    model = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)

    if strategy == "baseline":
        pass

    elif strategy == "reset_decoder":
        model.decoder.block.apply(model._init_weights)
        model.decoder.final_layer_norm.apply(model._init_weights)

    elif strategy == "reset_decoder_head":
        model.decoder.block.apply(model._init_weights)
        model.decoder.final_layer_norm.apply(model._init_weights)
        model.lm_head.apply(model._init_weights)

    elif strategy == "reset_decoder_last2":
        for layer in model.decoder.block[-2:]:
            layer.apply(model._init_weights)
        model.decoder.final_layer_norm.apply(model._init_weights)

    elif strategy == "perturb_decoder_last2_small":
        perturb_decoder_tail_layers(model, n=2, std_ratio=0.005)

    elif strategy == "perturb_decoder_last2_medium":
        perturb_decoder_tail_layers(model, n=2, std_ratio=0.01)

    elif strategy == "perturb_decoder_last2_large":
        perturb_decoder_tail_layers(model, n=2, std_ratio=0.02)

    elif strategy == "perturb_decoder_last3_large":
        perturb_decoder_tail_layers(model, n=3, std_ratio=0.02)

    elif strategy == "freeze_encoder":
        for parameter in model.encoder.parameters():
            parameter.requires_grad = False

    elif strategy == "freeze_encoder_reset_decoder":
        for parameter in model.encoder.parameters():
            parameter.requires_grad = False
        model.decoder.block.apply(model._init_weights)
        model.decoder.final_layer_norm.apply(model._init_weights)

    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    return model


# =====================================================
# Runtime setup
# =====================================================
set_random_seed(Config.SEED)

train_dataframe = pd.read_csv(os.path.join(Config.INPUT_DIR, Config.TRAIN_FILE))
print(f"Original Train Data: {len(train_dataframe)} docs")

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)
print("tokenizer load time:", time.time() - t0)


# =====================================================
# Build train dataset
# =====================================================
training_examples = build_training_examples(
    train_dataframe,
    keep_original=Config.KEEP_ORIGINAL,
    use_sentence_aligner=Config.USE_SENTENCE_ALIGNER,
    augment_ratio=Config.AUGMENT_RATIO
)

print(f"Final Train Data After Augmentation: {len(training_examples)} samples")
print(training_examples.head(5))

hf_train_dataset = Dataset.from_pandas(training_examples.reset_index(drop=True))
tokenized_train_dataset = hf_train_dataset.map(
    tokenize_translation_batch,
    batched=True,
    remove_columns=hf_train_dataset.column_names
)

gc.collect()
torch.cuda.empty_cache()


# =====================================================
# Train model
# =====================================================
strategy = "baseline"
model = load_model_with_strategy(strategy)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR,

    eval_strategy="no",
    save_strategy="epoch",

    learning_rate=Config.LEARNING_RATE,
    fp16=False,

    per_device_train_batch_size=Config.PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,

    weight_decay=0.05,
    save_total_limit=5,
    num_train_epochs=Config.EPOCHS,

    logging_steps=177,
    report_to="none",

    lr_scheduler_type="cosine",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Starting Full Training (single GPU)...")
trainer.train()

trainer.save_model(Config.OUTPUT_DIR)
tokenizer.save_pretrained(Config.OUTPUT_DIR)

print(f"Full model saved to {Config.OUTPUT_DIR}")